# H2O AutoML
**Extended — Pattern Recognition for the Rest of Us**

> Automated model training, stacking, and built-in explainability (SHAP + variable importance) using H2O.ai — runs free in Colab.

**Connection to Model Explainability:** H2O AutoML trains dozens of models automatically *and* produces SHAP values, partial dependence plots, and variable importance out of the box — making it one of the most practical ways to combine AutoML with explainability in a single workflow.

**How to use this notebook:**
- Run cells top to bottom (Runtime → Run all, or Shift+Enter per cell)
- Fill in the `# YOUR CODE HERE` sections in the exercise
- Complete the quiz at the bottom
- Save a copy to your GitHub fork: File → Save a copy in GitHub

In [ ]:
# Titanic via seaborn (bundled, no network needed in Colab)
import seaborn as sns
titanic = sns.load_dataset('titanic')
# Clean up
titanic['sex_num'] = (titanic['sex'] == 'female').astype(int)
titanic = titanic[['survived','pclass','sex_num','age','sibsp','parch','fare']].dropna()
print(f"Titanic: {titanic.shape}  |  Survival rate: {titanic['survived'].mean():.1%}")
titanic.head()

## 🎯 Concept & Intuition

H2O AutoML automates the most time-consuming parts of the modeling workflow:

- **Model training:** fits GBMs, XGBoost, Random Forests, Deep Learning, GLMs, and Stacked Ensembles automatically
- **Hyperparameter tuning:** searches parameter spaces per algorithm without manual grid search
- **Stacking:** builds a meta-learner on top of the best models, often beating any single model
- **Explainability:** generates SHAP values, variable importance, and PDP plots for any model in the leaderboard

Think of it as running LazyPredict + SHAP + cross-validation in one function call — with production-grade models.

**When to use H2O AutoML:**
- You need a strong baseline quickly on a new dataset
- You want to compare model families without writing boilerplate
- You need explainability alongside the model (stakeholder reporting, regulated industries)
- You're benchmarking before committing to a specific model architecture

## 🐍 Setup — Install & Initialize H2O

In [ ]:
# Install H2O (takes ~2 minutes in Colab — run once)
!pip install h2o -q

import h2o
from h2o.automl import H2OAutoML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Initialize H2O cluster
# In Colab this starts a local single-node cluster
h2o.init(max_mem_size='4G')
print("H2O version:", h2o.__version__)

## 📊 Load Data

We'll use the classic Titanic dataset — binary classification (survived or not). Swap in any CSV from the `../data/` folder.

## 🤖 Run H2O AutoML

In [ ]:
# Convert to H2O Frame
hf = h2o.H2OFrame(df)

# Mark target as categorical (classification)
hf['Survived'] = hf['Survived'].asfactor()

# Train/test split
train, test = hf.split_frame(ratios=[0.8], seed=42)

# Define predictors and target
target = 'Survived'
predictors = [c for c in hf.columns if c != target]

print(f"Training on {train.nrows} rows, testing on {test.nrows} rows")
print(f"Predictors: {predictors}")

In [ ]:
# Run AutoML
# max_models=15 trains up to 15 models (increase for better results, takes longer)
# seed ensures reproducibility
aml = H2OAutoML(
    max_models=15,
    seed=42,
    max_runtime_secs=120,   # 2-minute cap — increase for larger datasets
    sort_metric='AUC',
    include_algos=['GBM','XGBoost','RandomForest','GLM','StackedEnsemble']
)

aml.train(x=predictors, y=target, training_frame=train)

# View leaderboard — ranked by AUC
lb = aml.leaderboard
print("\n=== AutoML Leaderboard (top 10) ===")
lb.head(rows=10)

## 📈 Evaluate Best Model

In [ ]:
# Get best model
best = aml.leader
print("Best model:", best.model_id)
print("\n=== Performance on Test Set ===")
perf = best.model_performance(test)
print(f"AUC:      {perf.auc():.4f}")
print(f"Accuracy: {perf.accuracy()[0][1]:.4f}")
print(f"Logloss:  {perf.logloss():.4f}")

# Confusion matrix
print("\n=== Confusion Matrix ===")
print(perf.confusion_matrix())

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve, auc as sklearn_auc

# Get predictions on test set
preds = best.predict(test).as_data_frame()
test_df = test.as_data_frame()

fpr, tpr, _ = roc_curve(test_df['Survived'].astype(int), preds['p1'])
roc_auc = sklearn_auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#1e5fa8', lw=2, label=f'Best model (AUC = {roc_auc:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=.5, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — H2O AutoML Best Model')
ax.legend(loc='lower right')
ax.grid(True, alpha=.3)
plt.tight_layout()
plt.show()

## 🔬 Explainability — SHAP, Variable Importance & PDP

This is the key tie-in with the **Model Explainability** technique. H2O generates a full explanation report with one call.

In [ ]:
# Full explainability report — generates SHAP, variable importance, PDP
# This is the H2O equivalent of running SHAP + LIME + PDP separately
print("Generating explainability report...")
expl = best.explain(test)
# H2O renders the report inline in Colab automatically

In [ ]:
# Variable importance (standalone plot)
if hasattr(best, 'varimp'):
    varimp = best.varimp(use_pandas=True)
    if varimp is not None:
        fig, ax = plt.subplots(figsize=(7, 4))
        varimp_top = varimp.head(8)
        bars = ax.barh(varimp_top['variable'], varimp_top['scaled_importance'],
                       color='#1e5fa8', edgecolor='white', linewidth=.5)
        ax.set_xlabel('Scaled Importance')
        ax.set_title('Variable Importance — H2O Best Model')
        ax.invert_yaxis()
        for bar, val in zip(bars, varimp_top['scaled_importance']):
            ax.text(val + .005, bar.get_y() + bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=10, color='#1a1d23')
        ax.grid(True, axis='x', alpha=.3)
        plt.tight_layout()
        plt.show()
else:
    print("Variable importance not available for this model type (e.g. Stacked Ensemble).")
    print("Try: specific_model = h2o.get_model(aml.leaderboard[1,'model_id'])")

In [ ]:
# SHAP summary — individual prediction contributions
# Works on GBM, XGBoost, Random Forest models
try:
    # Get a non-ensemble model for SHAP (ensembles don't support SHAP directly)
    gbm_id = None
    for i in range(aml.leaderboard.nrows):
        mid = aml.leaderboard[i, 'model_id']
        if 'GBM' in mid or 'XGBoost' in mid:
            gbm_id = mid
            break

    if gbm_id:
        gbm_model = h2o.get_model(gbm_id)
        shap_values = gbm_model.shap_summary_plot(test)
        print(f"SHAP summary for: {gbm_id}")
    else:
        print("No GBM/XGBoost in leaderboard — adjust include_algos and rerun.")
except Exception as e:
    print(f"SHAP not available for this model: {e}")
    print("Tip: Run with include_algos=['GBM','XGBoost'] to ensure SHAP-compatible models.")

## 🔗 Connection: H2O AutoML ↔ Model Explainability

H2O AutoML and the standalone Model Explainability technique (SHAP/LIME) are complementary:

| | H2O AutoML | Standalone SHAP/LIME |
|---|---|---|
| **Best for** | Benchmarking many models quickly | Deep-dive on one specific model |
| **SHAP support** | Built-in for tree models | Full control, any model |
| **Stakeholder reports** | Auto-generated HTML | Custom plots |
| **Control** | Less (automated) | Full |

**Recommendation:** Use H2O AutoML to find the best model family, then apply standalone SHAP for the final stakeholder deliverable.

## 💼 Exercise

1. **Swap the dataset** — load a CSV from `../data/` or any sklearn dataset
2. **Change the target** — try a regression problem (set `asfactor()` for classification, leave as-is for regression)
3. **Increase `max_models`** to 25 — does the leaderboard change significantly?
4. **Compare model types** — find the best GBM vs the best Stacked Ensemble and compare AUC
5. **Generate a stakeholder memo** — using the variable importance and SHAP plots, write 3 sentences explaining which features drive survival predictions and what a decision-maker should know

```python
# YOUR CODE HERE
```

In [ ]:
# Exercise workspace
# YOUR CODE HERE

### Reflection

*In 2–3 sentences: when would you choose H2O AutoML over manually training sklearn models? What does the explainability output tell you that a single accuracy score doesn't?*

_..._

## 📚 Resources

- [H2O AutoML docs](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/automl.html)
- [H2O.ai free tutorials](https://www.h2o.ai/resources/)
- [H2O explain() documentation](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/explain.html)
- [ISLP Model Explainability notebook](../model-explainability.ipynb) — companion technique

## 🧪 Quiz

Answer in the code cell below — run it to check.

In [ ]:
answers = {
    "q1": "",  # What does H2O AutoML train automatically (name 3 model types)?
    "q2": "",  # What is a Stacked Ensemble and why does it often outperform single models?
    "q3": "",  # Name two explainability outputs H2O's explain() produces
    "q4": "",  # When would you use H2O AutoML vs standalone SHAP/LIME?
    "q5": "",  # One real-world use case where AutoML + explainability matters most
}

missing = [k for k,v in answers.items() if not v.strip()]
if missing:
    print(f"Still empty: {missing}")
else:
    print("All questions answered ✓")
    print("Save this notebook to your GitHub fork: File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "H2O AutoML"
# @title 🤖 AI-Graded Submission — fill in the box and click ▶ Run
# @markdown ---
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2 (one-time):** Get a free AI grading key
# @markdown - Go to [aistudio.google.com](https://aistudio.google.com) — use your **personal Gmail** (not university email — many universities block AI Studio)
# @markdown - Click **Get API key → Create API key** → copy it
# @markdown - In Colab: click the **🔑 key icon** in the left sidebar → Add secret → Name: `GEMINI_API_KEY` → paste key → toggle ON
# @markdown - Done — this persists across all 30 notebooks automatically
# @markdown
# @markdown **Step 3:** Click ▶ Run on this cell for instant AI feedback

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ── DO NOT EDIT BELOW THIS LINE ──────────────────────────────────────────────
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Try to explain your reasoning in 1-2 sentences."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback on your answers."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with good detail. "
                             "Add a free Gemini key for AI analysis of your specific reasoning."),
                "tip": "Get a free key at aistudio.google.com using your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             f"Complete the remaining {n_total - n_answered} questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"\n  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    print(f"  Student  : @{username}" if username else
          "  Student  : \u26a0\ufe0f  Enter your GitHub username in the box above")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...\n")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)\n")
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 choose your fork\n")


---
*Pattern Recognition for the Rest of Us · H2O AutoML · Extended*

*Tip: Shut down the H2O cluster when done to free memory:*
```python
h2o.shutdown(prompt=False)
```